In [ ]:
# !pip install -q --upgrade bitsandbytes==0.48.2 trl==0.25.1

In [5]:
import os
import math
from tqdm import tqdm
from huggingface_hub import login
import torch
import transformers
from transformers import GPT2Tokenizer, AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset, Dataset, DatasetDict
import wandb
from peft import PeftModel, LoraConfig
from trl import SFTTrainer, SFTConfig
from datetime import datetime
import matplotlib.pyplot as plt
from dotenv import load_dotenv
import numpy as np
from torch.nn import functional as F

## Colab notebook related

In [2]:
device = "cpu"
if torch.cuda.is_available():
    device = "cuda"
# elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
#     device = "mps"
print(f"using device: {device}")
device_type = "cuda" if device.startswith("cuda") else "cpu"

using device: cpu


## Local related

In [3]:
load_dotenv()
hf_token = os.getenv("HF_TOKEN")

In [ ]:
import sys
from pathlib import Path

cwd = Path.cwd().resolve()
project_root = next(
    (path for path in [cwd, *cwd.parents] if (path / "model" / "BabyGPT.py").exists()),
    cwd,
)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from model._BabyGPT import BabyGPT, BabyGPTConfig

## Get model

In [ ]:
hf_token = ""  # removed in public repo
login(hf_token, add_to_git_credential=True)

In [4]:
BASE_MODEL = "littleBallOfFur/baby-gpt-base"
FINETUNE_MODEL = "littleBallOfFur/baby-gpt-sft-tinystories-0424"
REVISION = None #"a33adb36da51a04e67a20b8ab395326c484fe3b7"

In [6]:
base_model = AutoModelForCausalLM.from_pretrained(BASE_MODEL,trust_remote_code=True,)

orig_base_model = AutoModelForCausalLM.from_pretrained(BASE_MODEL,trust_remote_code=True,)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL,trust_remote_code=True,)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Loading weights:   0%|          | 0/149 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/149 [00:00<?, ?it/s]

#### Alternatively, try real gpt2

In [8]:
BASE_MODEL = "gpt2"
FINETUNE_MODEL = "littleBallOfFur/baby-gpt-sft-tinystories-0423"
REVISION = None

In [6]:
base_model = AutoModelForCausalLM.from_pretrained(BASE_MODEL)

orig_base_model = AutoModelForCausalLM.from_pretrained(BASE_MODEL)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

### Create the fine tuned model

In [7]:
REVISION = None
if REVISION:
   fine_model = PeftModel.from_pretrained(base_model, FINETUNE_MODEL, revision = REVISION)
else:
   fine_model = PeftModel.from_pretrained(base_model, FINETUNE_MODEL)

adapter_config.json:   0%|          | 0.00/992 [00:00<?, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/3.55M [00:00<?, ?B/s]

In [8]:
orig_base_model

BabyGPTForCausalLM(
  (transformer): ModuleDict(
    (wte): Embedding(50304, 768)
    (wpe): Embedding(1024, 768)
    (h): ModuleList(
      (0-11): 12 x Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): CausalSelfAttention(
          (c_attn): Linear(in_features=768, out_features=2304, bias=True)
          (c_proj): Linear(in_features=768, out_features=768, bias=True)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): MLP(
          (c_fc): Linear(in_features=768, out_features=3072, bias=True)
          (gelu): GELU(approximate='tanh')
          (c_proj): Linear(in_features=3072, out_features=768, bias=True)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50304, bias=False)
)

In [9]:
def test_generation(model, prompt, max_length,temperature=1.0,verbose=False, auto=True):
    xgen = tokenizer.encode(prompt, return_tensors='pt')
    model.eval()
    # torch.manual_seed(42)
    # if torch.cuda.is_available():
    #     torch.cuda.manual_seed(42)
    if not auto:
        while xgen.size(1) < max_length:
            # forward the model to get the logits
            with torch.no_grad():
                logits = model(xgen).logits 
                # take the logits at the last position
                logits = logits[:, -1, :]/temperature # (B, vocab_size)

                probs = F.softmax(logits, dim=-1)
                topk_probs, topk_indices = torch.topk(probs, 50, dim=-1)
                ix = torch.multinomial(topk_probs, 1) # (B, 1)
                xcol = torch.gather(topk_indices, -1, ix) # (B, 1)
                xgen = torch.cat((xgen, xcol), dim=1)

                # v, _ = torch.topk(logits, min(50, logits.size(-1)))
                # logits[logits < v[:, [-1]]] = -float('Inf')
                # probs = F.softmax(logits, dim=-1)
                # xcol = torch.multinomial(probs, num_samples=1)
                # xgen = torch.cat((xgen, xcol), dim=1)
        tokens = xgen[0].tolist()
        
    else:
        with torch.no_grad():
            tokens = model.generate(xgen, 
                                    do_sample=True, 
                                    top_k=50, 
                                    temperature=temperature,
                                    top_p=0.95,
                                    # repetition_penalty=1.1,
                                    max_length=max_length )[0]

    decoded = tokenizer.decode(tokens)
    if verbose: print(tokens)
    print(f"{decoded}")


In [11]:
prompt = "A squirrel jumps from a tree "
print(f"\ninput: '{prompt}'\n" )
print("base model:")
test_generation(model=orig_base_model,prompt=prompt, max_length=100,auto=False)
print("="*40)
print("\nfine tuned model:")
test_generation(model=fine_model,prompt=prompt,max_length=100,auto=False)


input: 'A squirrel jumps from a tree '

base model:
A squirrel jumps from a tree ________.
Answer The squirrel jumps.
C. He was in a tree
A. a tree was
B. a tree was
B. He was in a tree
C. both a tree and a tree was
C. A tree was both a tree and a tree was both a tree and a tree was both a tree and a tree was both a tree and a tree was both a tree and a tree was both a tree and a tree

fine tuned model:
A squirrel jumps from a tree 
and hangs on a stick
The other squirrel jumps from the tree 
(see the photo)
and goes on a walk
without noticing
the tree is there.
And here’s another squirrel jumping in the tree 
and it’s not noticing it
they are not noticing it
The rest of the squirrels get confused
they want to catch it to eat it
So this time the squirrel gets stuck in the tree


In [14]:
prompt = "Long long time ago there was a squirrel "
test_generation(model=fine_model,prompt=prompt,max_length=100,auto=True)

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Long long time ago there was a squirrel urn. The squirrel had a box full of toys. Inside the box was a little girl named Sam. Sam was very proud of his box and said, "Thank you! I found something special!"
Sally said, "Yes, I found something special!" Sam was excited. He found a big, red ball and got to try it. Sam laughed and said, "Good, it's so new!"
Sam had a surprise. He wanted
